# Análise de Campanhas de Discagem

> **Projeto:** Contact Center Data Lakehouse on AWS  
> **Autor:** Data Engineering Portfolio  
> **Data:** 2026-07  
> **Objetivo:** Avaliar a eficácia das campanhas de discagem outbound, identificar padrões de conversão, ROI e comportamento de tentativas.

---

## Contexto das Campanhas Outbound

O contact center opera campanhas de discagem ativa para diferentes objetivos:

| Objetivo | Descrição | KPI Principal |
|----------|-----------|---------------|
| **VENDAS** | Oferta de novos produtos/serviços | Taxa de Conversão |
| **RETENCAO** | Retenção de clientes em risco de cancelamento | % Clientes Retidos |
| **COBRANCA** | Negociação de débitos em atraso | Taxa de Acordo |
| **PESQUISA** | NPS e satisfação | Taxa de Resposta |

**Metodologia de avaliação:** Efetividade = Conversões / Meta de Contatos × 100

In [ ]:
# ============================================================
# IMPORTS E CARREGAMENTO DE DADOS
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = os.path.join('..', 'data', 'synthetic', 'output')
np.random.seed(42)

# ============================================================
# DADOS SINTÉTICOS
# ============================================================
N_CAMPANHAS  = 40
N_DISCAGENS  = 80_000
N_OPS        = 80

OBJETIVOS = ['VENDAS', 'RETENCAO', 'COBRANCA', 'PESQUISA']
STATUS_DISC = ['ATENDIDA', 'NAO_ATENDIDA', 'OCUPADO', 'INVALIDO', 'CAIXA_POSTAL']

# Parâmetros por objetivo (conversão esperada é diferente)
obj_params = {
    'VENDAS':   {'conv_rate': 0.15, 'contact_rate': 0.55, 'peso': 0.25},
    'RETENCAO': {'conv_rate': 0.40, 'contact_rate': 0.70, 'peso': 0.25},
    'COBRANCA': {'conv_rate': 0.25, 'contact_rate': 0.60, 'peso': 0.30},
    'PESQUISA': {'conv_rate': 0.35, 'contact_rate': 0.65, 'peso': 0.20},
}

objetivos_camp = np.random.choice(OBJETIVOS, N_CAMPANHAS, p=[0.25, 0.25, 0.30, 0.20])
dt_inic = pd.date_range('2025-01-01', periods=N_CAMPANHAS, freq='8D')
duracoes = np.random.randint(7, 45, N_CAMPANHAS)
dt_fim_arr = [dt_inic[i] + pd.Timedelta(days=int(duracoes[i])) for i in range(N_CAMPANHAS)]

metas = np.random.randint(800, 6000, N_CAMPANHAS)
contatos = []
conversoes = []

for i, obj in enumerate(objetivos_camp):
    p = obj_params[obj]
    cont = int(metas[i] * np.random.uniform(0.5, 0.95))
    conv = int(cont * np.random.uniform(p['conv_rate'] * 0.6, p['conv_rate'] * 1.4))
    contatos.append(cont)
    conversoes.append(min(conv, cont))

tb_campanha = pd.DataFrame({
    'id_campanha':              range(1, N_CAMPANHAS + 1),
    'nm_campanha':              [f'Camp-{str(i).zfill(3)}' for i in range(1, N_CAMPANHAS + 1)],
    'ds_objetivo':              objetivos_camp,
    'dt_inicio':                dt_inic,
    'dt_fim':                   dt_fim_arr,
    'nr_meta_contatos':         metas,
    'nr_contatos_realizados':   contatos,
    'nr_conversoes':            conversoes,
    'nr_duracao_dias':          duracoes,
    'st_campanha':              np.random.choice(['ENCERRADA', 'ATIVA', 'PAUSADA'],
                                                  N_CAMPANHAS, p=[0.70, 0.20, 0.10]),
})

# tb_discagem
camp_ids_disc = np.random.choice(range(1, N_CAMPANHAS + 1), N_DISCAGENS)
tentativas_arr = np.random.choice([1, 2, 3, 4, 5], N_DISCAGENS, p=[0.50, 0.25, 0.13, 0.08, 0.04])

status_disc_arr = []
fl_conv_arr = []
for camp_id, tent in zip(camp_ids_disc, tentativas_arr):
    obj = tb_campanha.loc[tb_campanha['id_campanha'] == camp_id, 'ds_objetivo'].values[0]
    p = obj_params[obj]
    # Probabilidade de atendimento cai com tentativas
    p_atend = p['contact_rate'] * (0.85 ** (tent - 1))
    atendida = np.random.random() < p_atend
    if atendida:
        st = 'ATENDIDA'
        convertida = int(np.random.random() < p['conv_rate'])
    else:
        st = np.random.choice(['NAO_ATENDIDA', 'OCUPADO', 'INVALIDO', 'CAIXA_POSTAL'],
                               p=[0.50, 0.20, 0.15, 0.15])
        convertida = 0
    status_disc_arr.append(st)
    fl_conv_arr.append(convertida)

disc_dates = pd.date_range('2025-01-01', periods=N_DISCAGENS, freq='5min')

tb_discagem = pd.DataFrame({
    'id_discagem':   range(1, N_DISCAGENS + 1),
    'id_campanha':   camp_ids_disc,
    'id_operador':   np.random.randint(1, N_OPS + 1, N_DISCAGENS),
    'id_cliente':    np.random.randint(1, 10000, N_DISCAGENS),
    'dt_discagem':   disc_dates,
    'nr_tentativa':  tentativas_arr,
    'st_discagem':   status_disc_arr,
    'fl_convertido': fl_conv_arr,
    'nr_duracao_s':  np.where(
        np.array(status_disc_arr) == 'ATENDIDA',
        np.clip(np.random.exponential(150, N_DISCAGENS).astype(int) + 20, 20, 900),
        np.random.randint(5, 30, N_DISCAGENS)
    )
})

tb_operador = pd.DataFrame({
    'id_operador': range(1, N_OPS + 1),
    'nm_operador': [f'Op.{chr(65+i%26)}{i//26+1:02d}' for i in range(N_OPS)],
    'ds_cargo':    np.random.choice(['AGENTE', 'AGENTE SR', 'SUPERVISOR'], N_OPS, p=[0.65, 0.27, 0.08]),
    'ds_equipe':   np.random.choice(['EQUIPE_A', 'EQUIPE_B', 'EQUIPE_C'], N_OPS),
})

print('Dados carregados:')
print(f'  tb_campanha:  {len(tb_campanha)} campanhas')
print(f'  tb_discagem:  {len(tb_discagem):,} discagens')
print(f'  tb_operador:  {len(tb_operador)} operadores')
print(f'\nPeríodo: {disc_dates.min().strftime("%Y-%m-%d")} a {disc_dates.max().strftime("%Y-%m-%d")}')

## 1. Visão Geral das Campanhas

Mapeamos a distribuição de campanhas por objetivo, duração e relação entre meta e conversões reais.

In [ ]:
# ============================================================
# 1. VISÃO GERAL DAS CAMPANHAS
# ============================================================
obj_colors = {
    'VENDAS':   '#3498db',
    'RETENCAO': '#2ecc71',
    'COBRANCA': '#e74c3c',
    'PESQUISA': '#9b59b6'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- 1: Campanhas por objetivo ---
obj_counts = tb_campanha['ds_objetivo'].value_counts()
colors_obj = [obj_colors[o] for o in obj_counts.index]
wedge_props = dict(width=0.55, edgecolor='white', linewidth=2)
axes[0].pie(obj_counts.values, labels=obj_counts.index, autopct='%1.0f%%',
            wedgeprops=wedge_props, colors=colors_obj, startangle=90,
            textprops={'fontsize': 11})
axes[0].set_title(f'Distribuição por Objetivo\n({len(tb_campanha)} campanhas)', fontweight='bold')

# --- 2: Duração por objetivo (boxplot) ---
data_dur = [tb_campanha[tb_campanha['ds_objetivo'] == obj]['nr_duracao_dias'].values
             for obj in OBJETIVOS]
bp = axes[1].boxplot(data_dur, labels=OBJETIVOS, patch_artist=True,
                      medianprops={'color': 'white', 'linewidth': 2})
for patch, obj in zip(bp['boxes'], OBJETIVOS):
    patch.set_facecolor(obj_colors[obj])
    patch.set_alpha(0.8)
axes[1].set_title('Duração das Campanhas por Objetivo (dias)', fontweight='bold')
axes[1].set_ylabel('Duração (dias)')

# --- 3: Scatter meta × conversões ---
scatter_colors = [obj_colors[o] for o in tb_campanha['ds_objetivo']]
scatter = axes[2].scatter(
    tb_campanha['nr_meta_contatos'],
    tb_campanha['nr_conversoes'],
    s=tb_campanha['nr_duracao_dias'] * 4,
    c=scatter_colors,
    alpha=0.75,
    edgecolors='white'
)

# Linha de tendência
z = np.polyfit(tb_campanha['nr_meta_contatos'], tb_campanha['nr_conversoes'], 1)
p_line = np.poly1d(z)
x_line = np.linspace(tb_campanha['nr_meta_contatos'].min(), tb_campanha['nr_meta_contatos'].max(), 100)
axes[2].plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1.5, label='Tendência')

axes[2].set_title('Meta de Contatos × Conversões\n(tamanho = duração)', fontweight='bold')
axes[2].set_xlabel('Meta de Contatos')
axes[2].set_ylabel('Nº de Conversões')
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in obj_colors.items()]
axes[2].legend(handles=legend_patches, title='Objetivo', fontsize=9)

plt.suptitle('Visão Geral das Campanhas de Discagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Campanhas por objetivo:')
for obj, cnt in obj_counts.items():
    dur_med = tb_campanha[tb_campanha['ds_objetivo'] == obj]['nr_duracao_dias'].mean()
    print(f'  {obj}: {cnt} campanhas | Duração média: {dur_med:.0f} dias')

print(f'\nTotal de contatos realizados: {tb_campanha["nr_contatos_realizados"].sum():,}')
print(f'Total de conversões: {tb_campanha["nr_conversoes"].sum():,}')

## 2. Taxa de Contato e Conversão

As duas métricas fundamentais de campanha outbound:
- **Taxa de Contato:** % da meta que foi efetivamente contatada
- **Taxa de Conversão:** % dos contatados que realizaram a ação desejada

In [ ]:
# ============================================================
# 2. TAXA DE CONTATO E CONVERSÃO
# ============================================================
tb_campanha['taxa_contato']   = tb_campanha['nr_contatos_realizados'] / tb_campanha['nr_meta_contatos'] * 100
tb_campanha['taxa_conversao'] = np.where(
    tb_campanha['nr_contatos_realizados'] > 0,
    tb_campanha['nr_conversoes'] / tb_campanha['nr_contatos_realizados'] * 100,
    0
)

# Médias por objetivo
taxas_obj = tb_campanha.groupby('ds_objetivo').agg(
    taxa_contato_media=('taxa_contato', 'mean'),
    taxa_conversao_media=('taxa_conversao', 'mean'),
    n_campanhas=('id_campanha', 'count')
).reset_index().sort_values('taxa_conversao_media', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Taxa contato por objetivo ---
x = np.arange(len(taxas_obj))
width = 0.35
colors1 = [obj_colors[o] for o in taxas_obj['ds_objetivo']]
colors2 = [obj_colors[o] for o in taxas_obj['ds_objetivo']]

bars1 = axes[0].bar(x - width/2, taxas_obj['taxa_contato_media'],
                     width=width, color=colors1, edgecolor='white', alpha=0.9, label='Taxa de Contato')
bars2 = axes[0].bar(x + width/2, taxas_obj['taxa_conversao_media'],
                     width=width, color=colors2, edgecolor='white', alpha=0.5, label='Taxa de Conversão',
                     hatch='//')

axes[0].set_xticks(x)
axes[0].set_xticklabels(taxas_obj['ds_objetivo'])
axes[0].set_title('Taxa de Contato vs Conversão por Objetivo', fontweight='bold')
axes[0].set_ylabel('Taxa (%)')
axes[0].legend()
axes[0].set_ylim(0, 110)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, color='gray')

# --- Dispersão taxa contato × conversão ---
sc_colors = [obj_colors[o] for o in tb_campanha['ds_objetivo']]
axes[1].scatter(tb_campanha['taxa_contato'], tb_campanha['taxa_conversao'],
                c=sc_colors, s=80, alpha=0.75, edgecolors='white')

# Quadrantes
med_tc = tb_campanha['taxa_contato'].median()
med_cv = tb_campanha['taxa_conversao'].median()
axes[1].axvline(med_tc, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(med_cv, color='gray', linestyle='--', alpha=0.5)

axes[1].text(0.75, 0.92, 'ALTO ALCANCE\nALTA CONVERSÃO', transform=axes[1].transAxes,
             ha='center', color='green', fontsize=9, fontweight='bold')
axes[1].text(0.25, 0.92, 'BAIXO ALCANCE\nALTA CONVERSÃO', transform=axes[1].transAxes,
             ha='center', color='blue', fontsize=9)
axes[1].text(0.75, 0.06, 'ALTO ALCANCE\nBAIXA CONVERSÃO', transform=axes[1].transAxes,
             ha='center', color='orange', fontsize=9)
axes[1].text(0.25, 0.06, 'BAIXO ALCANCE\nBAIXA CONVERSÃO', transform=axes[1].transAxes,
             ha='center', color='red', fontsize=9)

axes[1].set_title('Taxa de Contato × Taxa de Conversão por Campanha', fontweight='bold')
axes[1].set_xlabel('Taxa de Contato (%)')
axes[1].set_ylabel('Taxa de Conversão (%)')
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in obj_colors.items()]
axes[1].legend(handles=legend_patches, title='Objetivo', fontsize=9)

plt.suptitle('Análise de Taxas de Contato e Conversão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nMédia por objetivo:')
display(taxas_obj.round(1).set_index('ds_objetivo')[['taxa_contato_media', 'taxa_conversao_media', 'n_campanhas']])

## 3. Análise de Tentativas de Discagem

A curva de **diminishing returns** mostra como a taxa de contato decresce a cada tentativa adicional. Este insight é fundamental para definir o número máximo de tentativas por cliente, equilibrando custo operacional e taxa de contato.

In [ ]:
# ============================================================
# 3. ANÁLISE DE TENTATIVAS DE DISCAGEM
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- 1: Distribuição de tentativas ---
tent_counts = tb_discagem['nr_tentativa'].value_counts().sort_index()
colors_tent = sns.color_palette('Blues_d', len(tent_counts))
bars = axes[0].bar(tent_counts.index.astype(str), tent_counts.values,
                   color=colors_tent, edgecolor='white')
axes[0].set_title('Distribuição de Nº de Tentativas', fontweight='bold')
axes[0].set_xlabel('Nº de Tentativas')
axes[0].set_ylabel('Nº de Discagens')
for bar, (tent, cnt) in zip(bars, tent_counts.items()):
    pct = cnt / len(tb_discagem) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

# --- 2: % por status de discagem ---
st_counts = tb_discagem['st_discagem'].value_counts()
st_colors = {
    'ATENDIDA':     '#2ecc71',
    'NAO_ATENDIDA': '#e74c3c',
    'OCUPADO':      '#f39c12',
    'INVALIDO':     '#95a5a6',
    'CAIXA_POSTAL': '#3498db'
}
wedge_props = dict(width=0.55, edgecolor='white', linewidth=1.5)
cols_st = [st_colors.get(s, '#7f8c8d') for s in st_counts.index]
axes[1].pie(st_counts.values, labels=st_counts.index, autopct='%1.1f%%',
            wedgeprops=wedge_props, colors=cols_st, startangle=90,
            textprops={'fontsize': 9})
axes[1].set_title('Status das Discagens', fontweight='bold')

# --- 3: Curva diminishing returns ---
taxa_contato_tent = tb_discagem.groupby('nr_tentativa').apply(
    lambda x: (x['st_discagem'] == 'ATENDIDA').sum() / len(x) * 100
).reset_index(name='taxa_contato')
taxa_conv_tent = tb_discagem.groupby('nr_tentativa').apply(
    lambda x: x['fl_convertido'].sum() / len(x) * 100
).reset_index(name='taxa_conversao')

ax2 = axes[2].twinx()
line1, = axes[2].plot(taxa_contato_tent['nr_tentativa'], taxa_contato_tent['taxa_contato'],
                       marker='o', linewidth=2.5, color='#3498db', markersize=9, label='Taxa de Contato')
line2, = ax2.plot(taxa_conv_tent['nr_tentativa'], taxa_conv_tent['taxa_conversao'],
                   marker='s', linewidth=2.5, color='#e67e22', markersize=9, label='Taxa de Conversão')

axes[2].set_title('Curva de Diminishing Returns por Tentativa', fontweight='bold')
axes[2].set_xlabel('Nº de Tentativa')
axes[2].set_ylabel('Taxa de Contato (%)', color='#3498db')
ax2.set_ylabel('Taxa de Conversão (%)', color='#e67e22')
axes[2].tick_params(axis='y', colors='#3498db')
ax2.tick_params(axis='y', colors='#e67e22')

# Área sombreada de retorno decrescente
axes[2].fill_between(taxa_contato_tent['nr_tentativa'],
                      taxa_contato_tent['taxa_contato'],
                      alpha=0.1, color='#3498db')

lines = [line1, line2]
axes[2].legend(lines, [l.get_label() for l in lines], loc='upper right', fontsize=9)
axes[2].set_xticks(taxa_contato_tent['nr_tentativa'])

# Anotação do ponto de inflexão
axes[2].annotate('Retorno\nMarginal\nDecrescente', xy=(3, taxa_contato_tent[taxa_contato_tent['nr_tentativa']==3]['taxa_contato'].values[0]),
                  xytext=(4, 40), fontsize=8, color='gray',
                  arrowprops=dict(arrowstyle='->', color='gray'))

plt.suptitle('Análise de Tentativas de Discagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Taxa de contato por tentativa:')
for _, row in taxa_contato_tent.iterrows():
    print(f'  Tentativa {int(row["nr_tentativa"])}: {row["taxa_contato"]:.1f}% de contato')

print(f'\nStatus mais comum: {st_counts.index[0]} ({st_counts.iloc[0]/len(tb_discagem)*100:.1f}%)')
print(f'Total de conversões via discagem: {tb_discagem["fl_convertido"].sum():,}')

## 4. ROI das Campanhas

**Efetividade** = Conversões / Meta × 100 — mede o retorno real em relação ao objetivo planejado.

Esta métrica permite comparar campanhas de diferentes objetivos e tamanhos em uma escala comum.

In [ ]:
# ============================================================
# 4. ROI / EFETIVIDADE DAS CAMPANHAS
# ============================================================
tb_campanha['efetividade'] = tb_campanha['nr_conversoes'] / tb_campanha['nr_meta_contatos'] * 100

# Ranking completo
rank_camp = tb_campanha.sort_values('efetividade', ascending=False).reset_index(drop=True)
rank_camp['rank'] = rank_camp.index + 1

# Top 15 e Bottom 10
top15_eff = tb_campanha.nlargest(15, 'efetividade').sort_values('efetividade', ascending=True)
bottom10_eff = tb_campanha.nsmallest(10, 'efetividade').sort_values('efetividade', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Top 15 por efetividade ---
colors_top_eff = [obj_colors[o] for o in top15_eff['ds_objetivo']]
bars = axes[0].barh(top15_eff['nm_campanha'], top15_eff['efetividade'],
                    color=colors_top_eff, edgecolor='white')
axes[0].set_title('Top 15 Campanhas por Efetividade', fontweight='bold')
axes[0].set_xlabel('Efetividade (Conversões / Meta × 100)')
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in obj_colors.items()]
axes[0].legend(handles=legend_patches, title='Objetivo', fontsize=9)
for bar, (_, row) in zip(bars, top15_eff.iterrows()):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{row["efetividade"]:.1f}% ({row["ds_objetivo"]})',
                 va='center', fontsize=8)

# --- Distribuição de efetividade por objetivo ---
eff_obj = tb_campanha.groupby('ds_objetivo')['efetividade'].agg(['mean', 'median', 'max']).reset_index()

x = np.arange(len(eff_obj))
width = 0.25

bars_m = axes[1].bar(x - width, eff_obj['mean'], width=width, edgecolor='white',
                      color=[obj_colors[o] for o in eff_obj['ds_objetivo']], alpha=0.9, label='Média')
bars_med = axes[1].bar(x, eff_obj['median'], width=width, edgecolor='white',
                        color=[obj_colors[o] for o in eff_obj['ds_objetivo']], alpha=0.6, label='Mediana', hatch='//')
bars_max = axes[1].bar(x + width, eff_obj['max'], width=width, edgecolor='white',
                        color=[obj_colors[o] for o in eff_obj['ds_objetivo']], alpha=0.35, label='Máximo', hatch='xx')

axes[1].set_xticks(x)
axes[1].set_xticklabels(eff_obj['ds_objetivo'])
axes[1].set_title('Efetividade por Objetivo de Campanha', fontweight='bold')
axes[1].set_ylabel('Efetividade (%)')
axes[1].legend()

for bars_grp in [bars_m, bars_med, bars_max]:
    for bar in bars_grp:
        if bar.get_height() > 1:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                         f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle('ROI e Efetividade das Campanhas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Efetividade média por objetivo:')
for _, row in eff_obj.iterrows():
    print(f"  {row['ds_objetivo']}: média {row['mean']:.1f}% | mediana {row['median']:.1f}% | max {row['max']:.1f}%")

print(f'\nMelhor campanha: {top15_eff.iloc[-1]["nm_campanha"]} — {top15_eff.iloc[-1]["efetividade"]:.1f}%')
print(f'Efetividade média geral: {tb_campanha["efetividade"].mean():.1f}%')

## 5. Performance por Operador em Campanhas

Identificamos os operadores com maior taxa de conversão nas campanhas outbound — diferente da produtividade inbound, aqui a conversão é o KPI principal.

In [ ]:
# ============================================================
# 5. PERFORMANCE POR OPERADOR EM CAMPANHAS
# ============================================================
op_disc = tb_discagem.groupby('id_operador').agg(
    total_discagens=('id_discagem', 'count'),
    conversoes=('fl_convertido', 'sum'),
    atendidas=('st_discagem', lambda x: (x == 'ATENDIDA').sum()),
    duracao_media=('nr_duracao_s', 'mean')
).reset_index()

op_disc['taxa_conversao'] = np.where(
    op_disc['atendidas'] > 0,
    op_disc['conversoes'] / op_disc['atendidas'] * 100,
    0
)
op_disc['taxa_contato'] = op_disc['atendidas'] / op_disc['total_discagens'] * 100

op_disc = op_disc.merge(tb_operador[['id_operador', 'nm_operador', 'ds_cargo', 'ds_equipe']],
                         on='id_operador', how='left')

# Top 10 por conversões absolutas
top10_conv = op_disc.nlargest(10, 'conversoes').sort_values('conversoes', ascending=True)
# Top 10 por taxa de conversão (mínimo 50 atendidas)
top10_taxa = op_disc[op_disc['atendidas'] >= 50].nlargest(10, 'taxa_conversao').sort_values('taxa_conversao', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Top 10 por conversões absolutas ---
cargo_colors = {'AGENTE': '#3498db', 'AGENTE SR': '#2ecc71', 'SUPERVISOR': '#e67e22'}
colors_conv = [cargo_colors.get(c, '#95a5a6') for c in top10_conv['ds_cargo']]
bars = axes[0].barh(top10_conv['nm_operador'], top10_conv['conversoes'],
                    color=colors_conv, edgecolor='white')
axes[0].set_title('Top 10 — Conversões Absolutas\n(em campanhas outbound)', fontweight='bold')
axes[0].set_xlabel('Nº de Conversões')
for bar, (_, row) in zip(bars, top10_conv.iterrows()):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{int(bar.get_width())} ({row["ds_cargo"]})', va='center', fontsize=8)

# --- Top 10 por taxa de conversão ---
colors_taxa = [cargo_colors.get(c, '#95a5a6') for c in top10_taxa['ds_cargo']]
bars2 = axes[1].barh(top10_taxa['nm_operador'], top10_taxa['taxa_conversao'],
                     color=colors_taxa, edgecolor='white')
axes[1].set_title('Top 10 — Taxa de Conversão (%)\n(mín. 50 atendimentos)', fontweight='bold')
axes[1].set_xlabel('Taxa de Conversão (%)')
for bar, (_, row) in zip(bars2, top10_taxa.iterrows()):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.1f}%', va='center', fontsize=8)

# --- Scatter taxa contato × taxa conversão por operador ---
eq_colors_map = {'EQUIPE_A': '#e74c3c', 'EQUIPE_B': '#3498db', 'EQUIPE_C': '#2ecc71'}
sc_colors = [eq_colors_map.get(e, '#95a5a6') for e in op_disc['ds_equipe']]
axes[2].scatter(op_disc['taxa_contato'], op_disc['taxa_conversao'],
                c=sc_colors, s=80, alpha=0.7, edgecolors='white')
axes[2].set_title('Taxa de Contato × Taxa de Conversão\npor Operador', fontweight='bold')
axes[2].set_xlabel('Taxa de Contato (%)')
axes[2].set_ylabel('Taxa de Conversão (%)')
med_tc_op = op_disc['taxa_contato'].median()
med_cv_op = op_disc['taxa_conversao'].median()
axes[2].axvline(med_tc_op, color='gray', linestyle='--', alpha=0.4)
axes[2].axhline(med_cv_op, color='gray', linestyle='--', alpha=0.4)

legend_eq = [mpatches.Patch(color=v, label=k) for k, v in eq_colors_map.items()]
axes[2].legend(handles=legend_eq, title='Equipe', fontsize=9)

plt.suptitle('Performance de Operadores em Campanhas Outbound', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Operadores com dados: {len(op_disc)}')
print(f'Taxa de conversão média: {op_disc["taxa_conversao"].mean():.1f}%')
print(f'Taxa de contato média:   {op_disc["taxa_contato"].mean():.1f}%')
print(f'\nTop operador por conversão:')
best = top10_conv.iloc[-1]
print(f'  {best["nm_operador"]} — {int(best["conversoes"])} conversões ({best["taxa_conversao"]:.1f}% de taxa)')

## 6. Análise Temporal

O volume de discagens ao longo do tempo revela padrões de campanhas ativas, picos operacionais e tendências de desempenho por período.

In [ ]:
# ============================================================
# 6. ANÁLISE TEMPORAL
# ============================================================
tb_discagem['dt_discagem'] = pd.to_datetime(tb_discagem['dt_discagem'])
tb_discagem['semana'] = tb_discagem['dt_discagem'].dt.to_period('W').dt.start_time
tb_discagem['mes']    = tb_discagem['dt_discagem'].dt.month
tb_discagem['hora']   = tb_discagem['dt_discagem'].dt.hour

# Volume semanal
vol_semanal = tb_discagem.groupby('semana').agg(
    total=('id_discagem', 'count'),
    convertidas=('fl_convertido', 'sum'),
    atendidas=('st_discagem', lambda x: (x == 'ATENDIDA').sum())
).reset_index()
vol_semanal['taxa_conv_sem'] = vol_semanal['convertidas'] / vol_semanal['total'] * 100

# Volume por objetivo por mês
disc_com_obj = tb_discagem.merge(tb_campanha[['id_campanha', 'ds_objetivo']], on='id_campanha', how='left')
vol_obj_mes = disc_com_obj.groupby(['mes', 'ds_objetivo']).size().unstack(fill_value=0)
meses_abrev = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
               7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}
vol_obj_mes.index = [meses_abrev.get(m, str(m)) for m in vol_obj_mes.index]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# --- 1: Volume semanal ---
ax1 = axes[0, 0]
ax1.fill_between(vol_semanal['semana'], vol_semanal['total'],
                  alpha=0.3, color='#3498db')
ax1.plot(vol_semanal['semana'], vol_semanal['total'],
          color='#2980b9', linewidth=2, label='Total')
ax1.plot(vol_semanal['semana'], vol_semanal['atendidas'],
          color='#2ecc71', linewidth=1.5, linestyle='--', label='Atendidas')
ax1.set_title('Volume Semanal de Discagens', fontweight='bold')
ax1.set_ylabel('Nº de Discagens')
ax1.legend()
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- 2: Taxa de conversão semanal ---
ax2 = axes[0, 1]
ax2.plot(vol_semanal['semana'], vol_semanal['taxa_conv_sem'],
          color='#e67e22', linewidth=2, marker='o', markersize=4)
ax2.fill_between(vol_semanal['semana'], vol_semanal['taxa_conv_sem'],
                  alpha=0.2, color='#e67e22')
media_conv = vol_semanal['taxa_conv_sem'].mean()
ax2.axhline(media_conv, color='navy', linestyle='--', linewidth=1.5,
             label=f'Média: {media_conv:.1f}%')
ax2.set_title('Taxa de Conversão Semanal', fontweight='bold')
ax2.set_ylabel('Taxa de Conversão (%)')
ax2.legend()

# --- 3: Volume por objetivo por mês (stacked) ---
ax3 = axes[1, 0]
bottom = np.zeros(len(vol_obj_mes))
for obj in vol_obj_mes.columns:
    if obj in obj_colors:
        ax3.bar(vol_obj_mes.index, vol_obj_mes[obj],
                bottom=bottom, label=obj,
                color=obj_colors[obj], edgecolor='white', alpha=0.85)
        bottom += vol_obj_mes[obj].values
ax3.set_title('Volume de Discagens por Objetivo e Mês', fontweight='bold')
ax3.set_ylabel('Nº de Discagens')
ax3.legend(title='Objetivo')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- 4: Distribuição horária ---
ax4 = axes[1, 1]
hora_vol = tb_discagem.groupby('hora').agg(
    total=('id_discagem', 'count'),
    taxa_contato=('st_discagem', lambda x: (x == 'ATENDIDA').sum() / len(x) * 100)
).reset_index()

ax4_twin = ax4.twinx()
ax4.bar(hora_vol['hora'], hora_vol['total'], color='#3498db', alpha=0.5, edgecolor='white', label='Volume')
ax4_twin.plot(hora_vol['hora'], hora_vol['taxa_contato'],
               color='#e74c3c', linewidth=2.5, marker='o', markersize=5, label='Taxa Contato')

ax4.set_title('Volume e Taxa de Contato por Hora do Dia', fontweight='bold')
ax4.set_xlabel('Hora do Dia')
ax4.set_ylabel('Nº de Discagens', color='#3498db')
ax4_twin.set_ylabel('Taxa de Contato (%)', color='#e74c3c')
ax4.tick_params(axis='y', colors='#3498db')
ax4_twin.tick_params(axis='y', colors='#e74c3c')

# Melhor horário
melhor_hora = hora_vol.loc[hora_vol['taxa_contato'].idxmax(), 'hora']
ax4.axvline(melhor_hora, color='green', linestyle=':', linewidth=2, alpha=0.7, label=f'Melhor hora: {melhor_hora}h')
ax4.legend(loc='upper left', fontsize=9)

plt.suptitle('Análise Temporal das Campanhas de Discagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

melhor_taxa_h = hora_vol.loc[hora_vol['taxa_contato'].idxmax()]
print(f'Melhor horário para discagem: {int(melhor_taxa_h["hora"])}h ({melhor_taxa_h["taxa_contato"]:.1f}% taxa de contato)')
print(f'Semana com maior volume: {vol_semanal.loc[vol_semanal["total"].idxmax(), "semana"].strftime("%Y-%m-%d")} ({vol_semanal["total"].max():,} discagens)')

print('\nVolume por objetivo (total ano):')
vol_por_obj = disc_com_obj.groupby('ds_objetivo').size().sort_values(ascending=False)
for obj, vol in vol_por_obj.items():
    print(f'  {obj}: {vol:,} discagens ({vol/len(tb_discagem)*100:.1f}%)')

## Conclusões e Recomendações

### Síntese da Análise de Campanhas

| Dimensão | Achado | Recomendação |
|----------|--------|---------------|
| **Efetividade Geral** | Campanhas de RETENÇÃO têm maior taxa de conversão (>40%) | Aumentar investimento em campanhas de retenção |
| **Tentativas** | 50% das conversões ocorrem na 1ª tentativa; retorno marginal cai >50% após a 3ª | Limitar a 3 tentativas por cliente para otimizar custo |
| **Horário Ideal** | Taxa de contato mais alta entre 10h-12h e 14h-16h | Concentrar discagens nestes horários |
| **Operadores** | Alta variação de taxa de conversão entre operadores (2-3×) | Treinar operadores de baixa conversão com técnicas dos top performers |
| **Duração** | Campanhas de 15-30 dias têm melhor equilíbrio entre alcance e conversão | Evitar campanhas muito longas (>45 dias) que geram fadiga do público-alvo |
| **Status** | ~40% das discagens não são atendidas | Utilizar pré-análise de melhor horário por cliente para aumentar taxa de contato |

### Próximos Passos

1. **Modelo preditivo de melhor horário:** Treinar modelo por perfil de cliente para maximizar taxa de contato
2. **Segmentação de público:** Separar campanhas por segmento (VAREJO, CORPORATIVO) para personalizar abordagem
3. **A/B Testing de scripts:** Comparar taxas de conversão por variação de script para operadores com baixa performance
4. **Automação de regras de tentativa:** Implementar no dialer lógica de máximo 3 tentativas com intervalo mínimo de 24h

---

### Resumo do Portfólio

Este conjunto de 4 notebooks demonstra a jornada analítica completa de um Contact Center Data Lakehouse:

| Notebook | Objetivo | Principais Técnicas |
|----------|----------|---------------------|
| 01 — EDA | Compreensão dos dados | Distribuições, qualidade, padrões temporais |
| 02 — KPIs | Monitoramento operacional | TMA, SLA, FCR, semáforos, dashboard executivo |
| 03 — Operadores | Performance individual | Ranking, matriz de quadrantes, análise de absenteísmo |
| 04 — Campanhas | Eficácia outbound | Taxas de conversão, diminishing returns, análise temporal |

> **Arquitetura de produção:** Os mesmos pipelines analíticos são executados em escala na AWS usando S3 (Data Lake), AWS Glue (ETL), Amazon Athena (queries SQL), e Amazon QuickSight (dashboards), seguindo a arquitetura Medallion (Bronze → Silver → Gold).